# Week 8 — Neural ODEs & Physics-Informed Neural Networks (PINNs)

> **Differential Equations for Scientists & Engineers**  
> *Where differential equations meet deep learning — adjoint sensitivity and automatic differentiation from scratch.*

---

## Learning Objectives

1. Understand **Neural ODEs** as continuous-depth neural networks
2. Derive the **adjoint sensitivity method** for backpropagating through an ODE solver
3. Implement a minimal **Neural ODE** in PyTorch and train it on trajectory data
4. Understand **PINNs** (Physics-Informed Neural Networks) as a meshless PDE solver
5. Implement a PINN from scratch for the **1D Poisson** and **heat equation**
6. Compare data-driven vs physics-informed approaches


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

try:
    import torch
    import torch.nn as nn
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f"PyTorch {torch.__version__} available.")
except ImportError:
    HAS_TORCH = False
    print("PyTorch not found. Sections 2–3 require: pip install torch")

---

## 1. The Neural ODE Idea

A **Residual Network** block computes $\mathbf{h}_{t+1} = \mathbf{h}_t + f_{\theta}(\mathbf{h}_t)$, which is the **Euler discretisation** of the ODE:

$$\frac{d\mathbf{h}}{dt} = f_{\theta}(\mathbf{h}(t), t)$$

A **Neural ODE** replaces the discrete stack of residual layers with a **continuous dynamics network** and an ODE solver.

**Key insight:** The output $\mathbf{h}(T)$ is computed by an ODE solver. To train $\theta$, we need $\frac{\partial \mathcal{L}}{\partial \theta}$ — which requires differentiating through the ODE solver.

### 1.1 Adjoint Sensitivity Method

Define the **adjoint state** $\mathbf{a}(t) = \frac{\partial \mathcal{L}}{\partial \mathbf{h}(t)}$. It satisfies the **adjoint ODE**:

$$\frac{d\mathbf{a}}{dt} = -\mathbf{a}^\top \frac{\partial f_\theta}{\partial \mathbf{h}}$$

integrated **backwards** from $T$ to $0$. The gradient with respect to parameters is:

$$\frac{\partial \mathcal{L}}{\partial \theta} = -\int_T^0 \mathbf{a}(t)^\top \frac{\partial f_\theta}{\partial \theta}\,dt$$

This has **constant memory** (vs. $O(T)$ for BPTT) — the key advantage of the adjoint method.

In [ ]:
# --- Adjoint method demo in NumPy: scalar ODE, explicit gradient ---
# Problem: y' = theta * y, y(0) = 1, minimise L = (y(1) - y_target)^2
# Exact: y(t) = exp(theta * t), grad = 2 * (y(1) - y_target) * y(1)

def forward_euler(theta, t_end=1.0, N=1000):
    h = t_end / N
    y = 1.0
    for _ in range(N):
        y += h * theta * y
    return y

def adjoint_gradient(theta, y_target, t_end=1.0, N=1000):
    """
    Compute dL/dtheta via the adjoint method.
    Forward: y' = theta*y, y(0)=1
    L = (y(T) - y_target)^2
    Adjoint: discrete recurrence a_k = a_{k+1}*(1+h*theta)
    Grad:    dL/dtheta = sum_k a_{k+1} * y_{k-1} * h
    """
    h = t_end / N
    # Forward pass — store trajectory
    # Discrete update:  y_{k+1} = y_k * (1 + h*theta)
    y_traj = [1.0]
    y = 1.0
    for _ in range(N):
        y += h * theta * y
        y_traj.append(y)

    L = (y_traj[-1] - y_target)**2

    # Backward: DISCRETE adjoint — consistent with the forward update.
    # From y_{k+1} = y_k*(1 + h*theta):
    #   dy_{k+1}/dy_k   = (1 + h*theta)   -> adjoint recurrence
    #   dy_{k+1}/dtheta = y_k * h         -> parameter-gradient term
    a = 2 * (y_traj[-1] - y_target)        # dL/dy_N
    grad = 0.0
    for k in range(N, 0, -1):
        grad += a * (y_traj[k-1] * h)      # accumulate dL/dtheta
        a = a * (1 + h * theta)            # propagate adjoint backward

    return L, grad


# Verify gradient against finite differences
theta_test = 0.5
y_target = np.exp(1.0)   # true solution at theta=1

L, grad_adj = adjoint_gradient(theta_test, y_target)

eps = 1e-5
L_plus  = (forward_euler(theta_test + eps) - y_target)**2
L_minus = (forward_euler(theta_test - eps) - y_target)**2
grad_fd = (L_plus - L_minus) / (2 * eps)

print(f"Adjoint gradient:    {grad_adj:.8f}")
print(f"Finite-diff gradient:{grad_fd:.8f}")
print(f"Relative error:      {abs(grad_adj - grad_fd)/abs(grad_fd):.2e}")

# Simple gradient descent to recover theta
theta = 0.0
lr = 0.05
losses = []
thetas = [theta]

for step in range(300):
    L, grad = adjoint_gradient(theta, y_target)
    theta -= lr * grad
    losses.append(L)
    thetas.append(theta)

print(f"\nOptimised theta = {theta:.6f} (true = 1.0)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(losses, 'b-', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Adjoint-based Gradient Descent')

axes[1].plot(thetas, 'r-', lw=2)
axes[1].axhline(1.0, color='k', ls='--', label='True θ=1')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('θ')
axes[1].set_title('Parameter Recovery')
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 2. Neural ODE in PyTorch — Spiral Trajectory

In [ ]:
if HAS_TORCH:
    # Generate training data: true spiral dynamics
    # dx/dt = -y + x*(1 - x^2 - y^2)
    # dy/dt =  x + y*(1 - x^2 - y^2)
    def true_dynamics(state, t):
        x, y = state
        r2 = x**2 + y**2
        return np.array([-y + x*(1 - r2), x + y*(1 - r2)])

    def rk4_np(f, y0, t_arr):
        states = [y0]
        y = y0.copy()
        for i in range(len(t_arr)-1):
            h = t_arr[i+1] - t_arr[i]
            k1 = f(y, t_arr[i])
            k2 = f(y + h/2*k1, t_arr[i]+h/2)
            k3 = f(y + h/2*k2, t_arr[i]+h/2)
            k4 = f(y + h*k3,   t_arr[i]+h)
            y = y + h/6*(k1+2*k2+2*k3+k4)
            states.append(y.copy())
        return np.array(states)

    t_train = np.linspace(0, 6, 60)
    y0_train = np.array([2.0, 0.0])
    traj_true = rk4_np(true_dynamics, y0_train, t_train)

    # Neural ODE: learn dynamics from data
    class ODEFunc(nn.Module):
        """The learned RHS of the ODE: dh/dt = f_theta(h)"""
        def __init__(self, hidden=32):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, hidden),
                nn.Tanh(),
                nn.Linear(hidden, hidden),
                nn.Tanh(),
                nn.Linear(hidden, 2),
            )

        def forward(self, t, y):
            return self.net(y)

    def euler_solve_torch(func, y0, t_arr):
        """Simple Euler ODE solver in PyTorch (differentiable)."""
        ys = [y0]
        y = y0
        for i in range(len(t_arr)-1):
            dt = t_arr[i+1] - t_arr[i]
            dy = func(t_arr[i], y)
            y = y + dt * dy
            ys.append(y)
        return torch.stack(ys)

    # Training
    model = ODEFunc(hidden=48)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    t_tensor = torch.tensor(t_train, dtype=torch.float32)
    y0_tensor = torch.tensor(y0_train, dtype=torch.float32)
    traj_target = torch.tensor(traj_true, dtype=torch.float32)

    losses_node = []
    for epoch in range(500):
        optimizer.zero_grad()
        traj_pred = euler_solve_torch(model, y0_tensor, t_tensor)
        loss = torch.mean((traj_pred - traj_target)**2)
        loss.backward()
        optimizer.step()
        losses_node.append(loss.item())

    # Plot
    with torch.no_grad():
        traj_pred_np = euler_solve_torch(model, y0_tensor, t_tensor).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(traj_true[:, 0], traj_true[:, 1], 'k-', lw=2, label='True')
    axes[0].plot(traj_pred_np[:, 0], traj_pred_np[:, 1], 'r--', lw=2, label='Neural ODE')
    axes[0].set_title('Phase Space: True vs Neural ODE')
    axes[0].legend(frameon=False); axes[0].set_aspect('equal')

    axes[1].plot(t_train, traj_true[:, 0], 'k-', lw=2, label='True x(t)')
    axes[1].plot(t_train, traj_pred_np[:, 0], 'r--', lw=2, label='Predicted x(t)')
    axes[1].set_title('x-component vs time')
    axes[1].legend(frameon=False)

    axes[2].semilogy(losses_node, 'b-', lw=2)
    axes[2].set_title('Training Loss'); axes[2].set_xlabel('Epoch')
    plt.tight_layout(); plt.show()
else:
    print("Section 2 requires PyTorch. Install with: pip install torch")

---

## 3. Physics-Informed Neural Networks (PINNs)

A **PINN** trains a neural network $u_\theta(x)$ (or $u_\theta(x,t)$) to satisfy:

1. The **PDE residual** $\mathcal{L}_{pde}[u_\theta] = 0$ at collocation points
2. **Boundary** / **initial conditions** at boundary points

The total loss is:

$$\mathcal{L} = w_1\,\|\mathcal{L}_{pde}[u_\theta]\|^2 + w_2\,\|u_\theta - g\|^2_{\partial\Omega \cup \{t=0\}}$$

Derivatives of the network are computed via **automatic differentiation** (AD), so no mesh is needed.

### 3.1 PINN for the 1D Poisson Equation

$$-u''(x) = f(x) = \pi^2 \sin(\pi x), \quad x \in [0,1], \quad u(0) = u(1) = 0$$

Exact solution: $u(x) = \sin(\pi x)$.

In [ ]:
if HAS_TORCH:
    class PINN_Poisson(nn.Module):
        """Small MLP to approximate u(x) satisfying -u'' = f."""
        def __init__(self, hidden=50):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(1, hidden), nn.Tanh(),
                nn.Linear(hidden, hidden), nn.Tanh(),
                nn.Linear(hidden, hidden), nn.Tanh(),
                nn.Linear(hidden, 1),
            )

        def forward(self, x):
            return self.net(x)

    pinn = PINN_Poisson(hidden=50)
    opt = torch.optim.Adam(pinn.parameters(), lr=1e-3)

    # Collocation points (interior)
    x_col = torch.linspace(0, 1, 200).reshape(-1, 1).requires_grad_(True)
    # Boundary points
    x_bc  = torch.tensor([[0.0], [1.0]])
    u_bc  = torch.zeros(2, 1)

    f_source = lambda x: np.pi**2 * torch.sin(np.pi * x)

    losses_pinn = []
    for step in range(5000):
        opt.zero_grad()

        # PDE residual: -u'' - f = 0
        u_pred = pinn(x_col)
        u_x = torch.autograd.grad(u_pred.sum(), x_col,
                                   create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x_col,
                                    create_graph=True)[0]
        pde_residual = -u_xx - f_source(x_col)
        loss_pde = torch.mean(pde_residual**2)

        # Boundary condition loss
        loss_bc = torch.mean((pinn(x_bc) - u_bc)**2)

        loss = loss_pde + 10 * loss_bc
        loss.backward()
        opt.step()
        losses_pinn.append(loss.item())

    # Evaluation
    x_test = torch.linspace(0, 1, 300).reshape(-1, 1)
    with torch.no_grad():
        u_pinn = pinn(x_test).numpy().ravel()
    x_np = x_test.numpy().ravel()
    u_exact_poisson = np.sin(np.pi * x_np)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(x_np, u_exact_poisson, 'k-', lw=2, label='Exact $\\sin(\\pi x)$')
    axes[0].plot(x_np, u_pinn, 'r--', lw=2, label='PINN')
    axes[0].set_title('PINN Solution — 1D Poisson')
    axes[0].legend(frameon=False)

    axes[1].semilogy(np.abs(u_pinn - u_exact_poisson), 'steelblue', lw=2)
    axes[1].set_title('Pointwise Error')
    axes[1].set_xlabel('x index')

    axes[2].semilogy(losses_pinn, 'purple', lw=2)
    axes[2].set_title('PINN Training Loss')
    axes[2].set_xlabel('Epoch')
    plt.tight_layout(); plt.show()
    print(f"Max error: {np.max(np.abs(u_pinn - u_exact_poisson)):.4f}")
else:
    print("Section 3 requires PyTorch.")

---

## 4. PINNs vs Classical Methods — A Comparison

| Aspect | Classical FD/FEM | PINN |
|---|---|---|
| **Mesh** | Required | Meshless |
| **Accuracy** | High (proven convergence) | Empirical, problem-dependent |
| **Complex geometry** | Hard | Natural (just sample points) |
| **Training cost** | None | High (optimisation loop) |
| **Inverse problems** | Requires reformulation | Natural (add parameter to network) |
| **Extrapolation** | Not possible | Possible (with caveats) |
| **Guarantees** | Error bounds available | Few theoretical guarantees |

PINNs excel at **inverse problems** — recovering parameters or source terms from sparse observations while enforcing the PDE as a soft constraint.

---

## 5. Exercises

1. **(Adjoint)** Extend the adjoint method to a 2D system $\dot{\mathbf{y}} = A\mathbf{y}$ and verify the gradient $\partial \mathcal{L}/\partial A$ via finite differences.

2. **(Neural ODE extrapolation)** Train a Neural ODE on $t \in [0, 3]$ and evaluate on $t \in [3, 6]$. How does extrapolation performance compare to a standard RNN trained on the same data?

3. **(PINN for heat equation)** Train a PINN to solve $u_t = u_{xx}$, $u(x,0) = \sin(\pi x)$, $u(0,t) = u(1,t) = 0$ on $[0,1]\times[0,1]$. Use the analytical solution $u = e^{-\pi^2 t}\sin(\pi x)$ to measure error.

4. **(Inverse PINN)** Given noisy observations of $u(x)$ satisfying $-u'' = \lambda\sin(\pi x)$ with unknown $\lambda$, train a PINN that learns $u$ **and** $\lambda$ simultaneously. Verify recovery of $\lambda$.

5. **(Collocation strategy)** Compare uniform vs. adaptive collocation point placement (concentrated near sharp gradients) for a PINN solving the Burgers equation. How does the number of required points scale to achieve a target accuracy?
